In [0]:
%pip install Faker

dbutils.library.restartPython()

In [0]:
import sys
from pathlib import Path

current = Path.cwd()

for path in [current, *current.parents]:
    if (path / "ingestion" / "delta_state.py").exists():
        sys.path.insert(0, str(path))
        print("Using import root:", path)
        break
else:
    raise RuntimeError("Could not find ingestion/delta_state.py")

In [0]:
import importlib
from datetime import date

import ingestion.delta_state as delta_state
import src.data_generator.daily_generation as daily_generation

importlib.reload(delta_state)
importlib.reload(daily_generation)

generate_daily_records = daily_generation.generate_daily_records
validate_daily_records = daily_generation.validate_daily_records

SIMULATION_DATE = date(2026, 8, 23)
SEED = 42

In [0]:
state = delta_state.load_generator_state(spark)

print("Generator state loaded")

In [0]:
print("users:", len(state["users"]))
print("user_state:", len(state["user_state"]))
print("models:", len(state["models"]))
print("devices:", len(state["devices"]))
print("plans:", len(state["plans"]))
print("recent conversations:", len(state["recent"]["conversations"]))
print("recent messages:", len(state["recent"]["messages"]))
print("subscriptions:", len(state["subscriptions"]))
print("count entities:", sorted(state["counts"].keys()))

In [0]:
print("user signup tz:", state["users"][0]["signup_at"].tzinfo)

print(
    "message created tz:",
    state["recent"]["messages"][0]["created_at"].tzinfo,
)

print(
    "subscription start tz:",
    state["subscriptions"][0]["started_at"].tzinfo,
)

In [0]:
state_1 = delta_state.load_generator_state(spark)

first = generate_daily_records(
    state_1,
    SIMULATION_DATE,
    SEED,
)

state_2 = delta_state.load_generator_state(spark)

second = generate_daily_records(
    state_2,
    SIMULATION_DATE,
    SEED,
)

print("Both runs generated")

In [0]:
ENTITIES = [
    "user_updates",
    "conversations",
    "messages",
    "completions",
    "model_inferences",
    "feedback",
    "errors",
    "subscriptions",
    "purchases",
    "payments",
]

all_match = True

for entity in ENTITIES:
    match = first[entity] == second[entity]

    print(
        f"{entity}: "
        f"{len(first[entity]):,} / "
        f"{len(second[entity]):,} "
        f"{'MATCH' if match else 'DIFFERENT'}"
    )

    if not match:
        all_match = False

assert all_match

print("\nPASS: full daily generation is deterministic")

In [0]:
first_validation = validate_daily_records(
    first,
    state_1,
    SIMULATION_DATE,
)

second_validation = validate_daily_records(
    second,
    state_2,
    SIMULATION_DATE,
)

print("First validation:", first_validation)
print("Second validation:", second_validation)

assert first_validation == {
    "referential_violations": 0,
    "causal_violations": 0,
}

assert second_validation == {
    "referential_violations": 0,
    "causal_violations": 0,
}

print("PASS: both generated datasets validated")

In [0]:
for entity in ENTITIES:
    print(f"{entity}: {len(first[entity]):,}")

print(
    "\nActive users:",
    len(first["_metadata"]["active_user_ids"]),
)

print(
    "Continued conversations:",
    len(first["_metadata"]["continued_conversation_ids"]),
)

In [0]:
bronze_first, manifest_first = daily_generation._bronze_records(
    first,
    SEED,
    SIMULATION_DATE,
    state_1,
)

bronze_second, manifest_second = daily_generation._bronze_records(
    second,
    SEED,
    SIMULATION_DATE,
    state_2,
)

In [0]:
bronze_match = True

for entity in bronze_first:
    match = bronze_first[entity] == bronze_second[entity]

    print(
        f"{entity}: "
        f"clean={len(first[entity]):,}, "
        f"bronze={len(bronze_first[entity]):,}, "
        f"{'MATCH' if match else 'DIFFERENT'}"
    )

    if not match:
        bronze_match = False

assert bronze_match
assert manifest_first == manifest_second

print(
    "\nInjected quality issues:",
    len(manifest_first),
)

print("PASS: Bronze defect injection is deterministic")